In [1]:
import requests
import pandas as pd
import time
from pathlib import Path

In [2]:
# Create directories
PROJECT_ROOT = Path("..")

RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

RAW_DATA.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)

In [3]:
# -ptosis of interest
cell_death_terms = [
    "apoptosis",
    "anoikis",
    "necroptosis",
    "pyroptosis",
    "ferroptosis",
    "cuproptosis",
    "parthanatos",
    "paraptosis",
    "autophagy-dependent cell death",
    "autosis",
    "lysosome-dependent cell death",
    "entosis",
    "methuosis",
    "oxeiptosis",
    "disulfidptosis",
    "alkaliptosis",
    "NETosis",
    "PANoptosis"
]

In [4]:
UNIPROT_URL = "https://rest.uniprot.org/uniprotkb/search"

In [5]:
# Queries any given key word and returns genes related to it that are human and reviewed
def query_uniprot(keyword, reviewed=True, organism="9606"):
    
    query = f'"{keyword}" AND organism_id:{organism}'
    
    if reviewed:
        query += " AND reviewed:true"
    
    params = {
        "query": query,
        "format": "json",
        "size": 500
    }
    
    response = requests.get(
        UNIPROT_URL,
        params=params
    )
    
    response.raise_for_status()
    
    return response.json()

In [6]:
# Test with Apoptosis
apoptosis_results = query_uniprot("apoptosis")

len(apoptosis_results["results"])

500

In [7]:
# Creates table with details on each gene
def extract_uniprot_genes(results, death_type):
    
    genes = []
    
    for protein in results["results"]:
        
        accession = protein["primaryAccession"]
        
        # Protein name
        protein_name = (
            protein["proteinDescription"]
            ["recommendedName"]
            ["fullName"]
            ["value"]
        )
        
        # Gene name
        gene_name = None
        
        if "genes" in protein:
            gene_name = (
                protein["genes"][0]
                ["geneName"]
                ["value"]
            )
        
        genes.append({
            "gene": gene_name,
            "uniprot_id": accession,
            "protein_name": protein_name,
            "cell_death": death_type
        })
    
    return pd.DataFrame(genes)

In [8]:
# Test Apoptosis query results and inspect
apoptosis_df = extract_uniprot_genes(
    apoptosis_results,
    "apoptosis"
)

apoptosis_df.head()

,gene,uniprot_id,protein_name,cell_death
0,PAWR,Q96IZ0,PRKC apoptosis WT1 regulator protein,apoptosis
1,BIRC5,O15392,Baculoviral IAP repeat-containing protein 5,apoptosis
2,ATG5,Q9H1Y0,Autophagy protein 5,apoptosis
3,AIFM3,Q96NN9,Apoptosis-inducing factor 3,apoptosis
4,BCL2,P10415,Apoptosis regulator Bcl-2,apoptosis


In [9]:
# Testing quality of query strategy
apoptosis_df["gene"].nunique()

apoptosis_df.head(20)

,gene,uniprot_id,protein_name,cell_death
0,PAWR,Q96IZ0,PRKC apoptosis WT1 regulator protein,apoptosis
1,BIRC5,O15392,Baculoviral IAP repeat-containing protein 5,apoptosis
2,ATG5,Q9H1Y0,Autophagy protein 5,apoptosis
3,AIFM3,Q96NN9,Apoptosis-inducing factor 3,apoptosis
4,BCL2,P10415,Apoptosis regulator Bcl-2,apoptosis
5,BCL2L1,Q07817,Bcl-2-like protein 1,apoptosis
6,EAF2,Q96CJ1,ELL-associated factor 2,apoptosis
7,BCL2L14,Q9BZR8,Apoptosis facilitator Bcl-2-like protein 14,apoptosis
8,CCAR2,Q8N163,Cell cycle and apoptosis regulator protein 2,apoptosis
9,NAIF1,Q69YI7,Nuclear apoptosis-inducing factor 1,apoptosis


In [10]:
# Automate all cell death terms
all_results = []

for death_type in cell_death_terms:
    
    print(f"Querying {death_type}")
    
    results = query_uniprot(death_type)
    
    df = extract_uniprot_genes(
        results,
        death_type
    )
    
    all_results.append(df)
    
    time.sleep(1)

cell_death_long = pd.concat(
    all_results,
    ignore_index=True
)

Querying apoptosis
Querying anoikis
Querying necroptosis
Querying pyroptosis
Querying ferroptosis
Querying cuproptosis
Querying parthanatos
Querying paraptosis
Querying autophagy-dependent cell death
Querying autosis
Querying lysosome-dependent cell death
Querying entosis
Querying methuosis
Querying oxeiptosis
Querying disulfidptosis
Querying alkaliptosis
Querying NETosis
Querying PANoptosis


In [11]:
cell_death_long.head()

,gene,uniprot_id,protein_name,cell_death
0,PAWR,Q96IZ0,PRKC apoptosis WT1 regulator protein,apoptosis
1,BIRC5,O15392,Baculoviral IAP repeat-containing protein 5,apoptosis
2,ATG5,Q9H1Y0,Autophagy protein 5,apoptosis
3,AIFM3,Q96NN9,Apoptosis-inducing factor 3,apoptosis
4,BCL2,P10415,Apoptosis regulator Bcl-2,apoptosis


In [12]:
cell_death_long.shape

(756, 4)

In [13]:
cell_death_long["cell_death"].value_counts()

cell_death
apoptosis                         500
pyroptosis                        121
ferroptosis                        45
anoikis                            41
necroptosis                        24
PANoptosis                         11
autophagy-dependent cell death      5
parthanatos                         4
NETosis                             3
cuproptosis                         2
Name: count, dtype: int64

In [14]:
# Drop identical rows
cell_death_long = cell_death_long.drop_duplicates()

In [15]:
cell_death_long.shape

(756, 4)

In [16]:
# Save long-format table
cell_death_long.to_csv(
    RAW_DATA / "cell_death_regulators_long.csv",
    index=False
)

In [17]:
# Create one-hot encoded matrix
cell_death_matrix = (
    cell_death_long
    .assign(value=1)
    .pivot_table(
        index="gene",
        columns="cell_death",
        values="value",
        fill_value=0
    )
    .reset_index()
)

In [18]:
cell_death_matrix.head()

cell_death,gene,NETosis,PANoptosis,anoikis,apoptosis,autophagy-dependent cell death,cuproptosis,ferroptosis,necroptosis,parthanatos,pyroptosis
0,AATF,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,AATK,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,ABL1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,ACIN1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,ACLY,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [19]:
# counts hits and filters rows by descending order
cell_death_matrix["num_cell_death_programs"] = (
    cell_death_matrix
    .drop(columns=["gene"])
    .sum(axis=1)
)

cell_death_matrix = (
    cell_death_matrix
    .sort_values(
        by="num_cell_death_programs",
        ascending=False
    )
    .reset_index(drop=True)
)

In [20]:
# Save matrix
cell_death_matrix.to_csv(
    PROCESSED_DATA / "cell_death_regulator_matrix.csv",
    index=False
)

In [21]:
# Top 30 genes
cell_death_matrix.sort_values(
    "num_cell_death_programs",
    ascending=False
).head(30)

cell_death,gene,NETosis,PANoptosis,anoikis,apoptosis,autophagy-dependent cell death,cuproptosis,ferroptosis,necroptosis,parthanatos,pyroptosis,num_cell_death_programs
0,SQSTM1,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,4.0
1,CASP8,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,4.0
2,NINJ1,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,4.0
3,CASP6,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,4.0
4,MEFV,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,3.0
5,CASP1,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0
6,AIM2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,3.0
7,MAP3K7,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,3.0
8,ZBP1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,3.0
9,PYCARD,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0
